# Week 4 — Chunking 전략 비교 실험 (Chunking Experiments)

## 목표
3주차 baseline과 동일한 golden set으로 **chunking 전략 3종을 비교**해, 어떤 전략이 본 도메인(한·영 혼합 유방암 가이드라인)에 가장 잘 맞는지를 RAGAS 3지표로 수치화한다.

선행 노트북: `notebooks/week4_data_analysis.ipynb` (데이터 진단)
선행 문서: `docs/week4_retrospective.md` (전략 선택 근거)

## 비교 전략 (회고에서 확정)
| 전략 | 구성 | 근거 |
|---|---|---|
| **A (Baseline)** | RecursiveCharacterTextSplitter 512/100 | 3주차 그대로 (비교 기준점) |
| **B** | RecursiveCharacterTextSplitter 1000/200 | 한국어 페이지당 토큰 분포(평균 947)에 맞춰 chunk_size 키움 |
| **C** | Noise-cleaned + 언어별 차등 size Recursive (ko 900 / en 512) | 머리말·페이지번호 제거 + 한·영 토큰 밀도 2.4배 차이 반영 |

## 비교 설계 (변수 통제)
- A → B: **chunk_size 효과만** 격리 (동일 parser/embedding/LLM/retriever)
- B → C: 여기에 **노이즈 제거 + 언어별 size** 추가 효과
- embedding/LLM/TOP_K는 baseline과 동일로 고정 (chunking 외 변수 차단)


---
## 0. 패키지 (3주차와 동일 — 이미 설치됐으면 건너뛰기)

In [ ]:
# 3주차에서 이미 설치했다면 실행 불필요
# %pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-ollama
# %pip install -q sentence-transformers pymupdf chromadb ragas datasets pandas tiktoken

---
## 1. 경로 / 설정 (CONFIG)

baseline 값은 3주차 노트북과 **똑같이** 고정. 전략 B/C 파라미터만 여기서 조정한다.

In [2]:
from pathlib import Path
import os, json

# 3주차와 동일한 경로 구조
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"
for p in [DATA_PROCESSED, DATA_EVAL, VECTOR_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ---- baseline 고정값 (3주차와 동일) ----
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
OLLAMA_MODEL = "qwen3:4b"
EMBED_DEVICE = "cpu"            # GPU 있으면 "cuda" (인덱싱 속도 큰 차이)
TOP_K = 5

# ---- 전략별 chunking 파라미터 ----
STRATEGIES = {
    "A_baseline": {"kind": "recursive", "chunk_size": 512,  "chunk_overlap": 100},
    "B_size1000": {"kind": "recursive", "chunk_size": 1000, "chunk_overlap": 200},
    "C_cleaned":  {"kind": "cleaned",
                   "size_by_lang": {"ko": 900, "en": 512, "unknown": 800},
                   "overlap_by_lang": {"ko": 180, "en": 100, "unknown": 150}},
}

# 전략 A는 3주차 결과(week3_ragas_scores.csv)가 있으면 재사용 -> 재생성/재평가 생략 (시간 절약)
REUSE_WEEK3_FOR_A = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("strategies:", list(STRATEGIES.keys()))

PROJECT_ROOT: d:\rag_agent_project\rag-agent-portfolio
strategies: ['A_baseline', 'B_size1000', 'C_cleaned']


---
## 2. 공통 — PDF 로드 (3주차 로더 재사용)

모든 전략이 동일한 원본 Document에서 출발해야 비교가 공정하다.

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
import re

manifest_path = DATA_RAW / "metadata" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    meta_lookup = {m["filename"]: m for m in manifest if m.get("downloaded")}
else:
    meta_lookup = {}

def load_all_pdfs(pdf_root: Path) -> list:
    all_docs = []
    for pdf_path in pdf_root.rglob("*.pdf"):
        loader = PyMuPDFLoader(str(pdf_path))
        docs = loader.load()
        extra = meta_lookup.get(pdf_path.name, {})
        for d in docs:
            d.metadata.update({
                "filename": pdf_path.name,
                "source_folder": pdf_path.parent.name,
                "org": extra.get("org", pdf_path.parent.name),
                "title": extra.get("title", pdf_path.stem),
                "language": extra.get("language", "unknown"),
                "doc_type": extra.get("doc_type", "unknown"),
                "priority": extra.get("priority", "unknown"),
            })
        all_docs.extend(docs)
    return all_docs

documents = load_all_pdfs(DATA_RAW / "pdf")
print(f"로드된 페이지 Document 수: {len(documents)}")

# language 메타가 unknown인 경우 한글 비율로 fallback
def guess_lang(text: str) -> str:
    kr = len(re.findall(r"[\uac00-\ud7a3]", text))
    return "ko" if kr > 20 else "en"
for d in documents:
    if d.metadata.get("language") in (None, "unknown", "?"):
        d.metadata["language"] = guess_lang(d.page_content)

c:\Users\hfdt\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


로드된 페이지 Document 수: 796


---
## 3. 노이즈 클린 유틸 (전략 C 전용)

회고에서 지적한 두 문제를 보완한 버전이다.

- **문제 1 (페이지번호 과대집계)**: 숫자 줄 전부를 지우면 표 안 숫자까지 삭제됨 → **페이지 맨위/맨아래 위치**의 숫자 줄만 제거
- **문제 2 (본문 오제거)**: "3페이지 이상" 기준은 본문 bullet까지 지움 → **전체 페이지의 30% 이상에 반복**되는 줄만 머리말/꼬리말로 간주

In [4]:
from collections import Counter

PAGE_NUM_PATTERNS = [
    re.compile(r"^\d{1,4}$"),
    re.compile(r"^-\s?\d{1,4}\s?-$"),
    re.compile(r"^Page\s+\d+", re.IGNORECASE),
    re.compile(r"^\d+\s*/\s*\d+$"),
]

def detect_running_headers(doc_pages: list, min_ratio: float = 0.30, max_len: int = 50) -> set:
    """한 문서 내에서 전체 페이지의 min_ratio 이상에 반복되는 짧은 줄 = 머리말/꼬리말 후보."""
    n_pages = len(doc_pages)
    if n_pages < 4:
        return set()
    counter = Counter()
    for text in doc_pages:
        lines = {l.strip() for l in text.split("\n") if 1 <= len(l.strip()) <= max_len}
        for l in lines:
            counter[l] += 1
    threshold = max(3, int(n_pages * min_ratio))
    return {l for l, c in counter.items() if c >= threshold}

def strip_page_number_lines(text: str) -> str:
    """페이지 맨위/맨아래 위치의 숫자 전용 줄만 제거 (표 안 숫자 보존)."""
    lines = text.split("\n")
    nonempty_idx = [i for i, l in enumerate(lines) if l.strip()]
    if not nonempty_idx:
        return text
    edge = set(nonempty_idx[:2] + nonempty_idx[-2:])  # 앞 2줄, 끝 2줄
    out = []
    for i, l in enumerate(lines):
        s = l.strip()
        if i in edge and any(p.match(s) for p in PAGE_NUM_PATTERNS):
            continue
        out.append(l)
    return "\n".join(out)

def clean_documents(docs: list) -> list:
    """filename 단위로 머리말 감지 -> 머리말·페이지번호 제거."""
    by_file = {}
    for d in docs:
        by_file.setdefault(d.metadata.get("filename", "?"), []).append(d)

    cleaned = []
    total_removed = 0
    for fname, pages in by_file.items():
        headers = detect_running_headers([p.page_content for p in pages])
        for d in pages:
            text = strip_page_number_lines(d.page_content)
            kept = []
            for l in text.split("\n"):
                if l.strip() in headers:
                    total_removed += 1
                    continue
                kept.append(l)
            cleaned.append(Document(page_content="\n".join(kept), metadata=dict(d.metadata)))
    print(f"제거된 머리말/꼬리말 줄 수(누적): {total_removed}")
    return cleaned

# 감지된 머리말 샘플 확인 (ESMO)
sample_pages = [d.page_content for d in documents if d.metadata.get("filename","").startswith("esmo")]
print("ESMO 머리말/꼬리말 후보:", detect_running_headers(sample_pages))

ESMO 머리말/꼬리말 후보: {'환자를 위한 ESMO 안내서', '유방암'}


---
## 4. 청킹 전략 3종 정의

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def make_splitter(size: int, overlap: int) -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=SEPARATORS, length_function=len,
    )

def build_chunks(strategy_name: str) -> list:
    cfg = STRATEGIES[strategy_name]
    if cfg["kind"] == "recursive":
        return make_splitter(cfg["chunk_size"], cfg["chunk_overlap"]).split_documents(documents)
    if cfg["kind"] == "cleaned":
        cleaned = clean_documents(documents)
        out = []
        for lang in set(d.metadata.get("language", "unknown") for d in cleaned):
            size = cfg["size_by_lang"].get(lang, cfg["size_by_lang"]["unknown"])
            overlap = cfg["overlap_by_lang"].get(lang, cfg["overlap_by_lang"]["unknown"])
            sub = [d for d in cleaned if d.metadata.get("language", "unknown") == lang]
            out.extend(make_splitter(size, overlap).split_documents(sub))
        return out
    raise ValueError(strategy_name)

# 각 전략 chunk 통계 (인덱싱 전, 빠름)
chunk_store = {}
for name in STRATEGIES:
    ck = build_chunks(name)
    chunk_store[name] = ck
    avg = sum(len(c.page_content) for c in ck) / max(len(ck), 1)
    print(f"{name:12s} chunk수={len(ck):5d}  평균길이(문자)={avg:.0f}")

A_baseline   chunk수= 3214  평균길이(문자)=436
B_size1000   chunk수= 1753  평균길이(문자)=790
제거된 머리말/꼬리말 줄 수(누적): 1560
C_cleaned    chunk수= 2436  평균길이(문자)=556


---
## 5. 인덱싱 / retriever 빌더 (전략별 별도 컬렉션)

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)
print("embedding 로드 완료")

def build_retriever(strategy_name: str):
    """전략별 Chroma 컬렉션 생성(없으면) 또는 재사용."""
    vdir = VECTOR_ROOT / f"week4_{strategy_name}"
    vdir.mkdir(parents=True, exist_ok=True)
    coll = f"breast_rag_week4_{strategy_name}"
    client = chromadb.PersistentClient(path=str(vdir))
    if coll in [c.name for c in client.list_collections()]:
        vs = Chroma(collection_name=coll, embedding_function=embeddings, persist_directory=str(vdir))
        print(f"  {strategy_name}: 기존 컬렉션 재사용 ({vs._collection.count()}개)")
    else:
        ck = chunk_store[strategy_name]
        print(f"  {strategy_name}: 신규 인덱싱 {len(ck)}개 ... (CPU면 시간 소요)")
        vs = Chroma.from_documents(documents=ck, embedding=embeddings,
                                   collection_name=coll, persist_directory=str(vdir))
    return vs.as_retriever(search_kwargs={"k": TOP_K})

embedding 로드 완료


---
## 6. RAG 체인 (3주차와 동일 프롬프트)

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.0)

RAG_PROMPT = ChatPromptTemplate.from_template("""당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parts.append(f"[{i}] 출처: {m.get('org','?')} / {m.get('title','?')} / p.{m.get('page','?')}\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

def make_ask(retriever):
    def ask(question: str):
        docs = retriever.invoke(question)
        prompt = RAG_PROMPT.format(context=format_context(docs), question=question)
        answer = (llm | StrOutputParser()).invoke(prompt)
        return {"question": question, "answer": answer,
                "contexts": [d.page_content for d in docs]}
    return ask

---
## 7. 평가 질문 세트 (3주차 golden_set_v0 재사용)

3주차와 **동일한 질문 세트**를 써야 baseline 비교가 성립한다. (필요시 20개로 확장 — 5주차 golden set 고도화 때)

In [8]:
import pandas as pd

golden_path = DATA_EVAL / "golden_set_v0.csv"
if golden_path.exists():
    df_golden = pd.read_csv(golden_path)
    print(f"golden_set_v0 로드: {len(df_golden)}문항")
else:
    raise FileNotFoundError("golden_set_v0.csv 없음 — 3주차 노트북 Cell 19를 먼저 실행해 생성하세요.")
golden = df_golden.to_dict("records")
df_golden[["question"]]

FileNotFoundError: golden_set_v0.csv 없음 — 3주차 노트북 Cell 19를 먼저 실행해 생성하세요.

---
## 8. 전략별 실행 + RAGAS 평가

주의: qwen3:4b CPU 기준, 전략 1개당 생성 10문항 + RAGAS 평가로 수십 분 소요. B/C 두 전략만 돌려도 1시간 이상 걸릴 수 있다.
- 전략 A는 `REUSE_WEEK3_FOR_A=True`면 3주차 점수(`week3_ragas_scores.csv`)를 그대로 사용 → 재생성/재평가 생략
- GPU가 있으면 CONFIG의 `EMBED_DEVICE="cuda"`로 인덱싱 시간을 크게 줄일 수 있다

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm

ragas_llm = LangchainLLMWrapper(ChatOllama(model=OLLAMA_MODEL, temperature=0.0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]

def run_strategy(name: str) -> pd.DataFrame:
    retriever = build_retriever(name)
    ask = make_ask(retriever)
    rows = []
    for g in tqdm(golden, desc=f"RAG[{name}]"):
        r = ask(g["question"])
        rows.append({"question": r["question"], "answer": r["answer"],
                     "contexts": r["contexts"], "ground_truth": g.get("ground_truth", "")})
    ds = Dataset.from_list(rows)
    scores = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb)
    df = scores.to_pandas()
    df.to_csv(DATA_PROCESSED / f"week4_ragas_{name}.csv", index=False, encoding="utf-8-sig")
    return df

score_tables = {}
for name in STRATEGIES:
    if name == "A_baseline" and REUSE_WEEK3_FOR_A:
        wk3 = DATA_PROCESSED / "week3_ragas_scores.csv"
        if wk3.exists():
            score_tables[name] = pd.read_csv(wk3)
            print(f"{name}: 3주차 결과 재사용 ({wk3.name})")
            continue
        print(f"{name}: week3 결과 없음 -> 직접 실행")
    score_tables[name] = run_strategy(name)
    print(f"{name}: 완료")

---
## 9. 비교 결과표

In [ ]:
rows = []
for name in STRATEGIES:
    df = score_tables[name]
    ck = chunk_store[name]
    row = {"전략": name,
           "chunk수": len(ck),
           "평균길이": round(sum(len(c.page_content) for c in ck)/max(len(ck),1))}
    for col in METRIC_COLS:
        row[col] = round(df[col].mean(), 4) if col in df.columns else None
    rows.append(row)

df_compare = pd.DataFrame(rows)

# baseline 대비 델타 추가
base = df_compare[df_compare["전략"] == "A_baseline"].iloc[0]
for col in METRIC_COLS:
    df_compare[col + "_delta"] = (df_compare[col] - base[col]).round(4)

df_compare.to_csv(DATA_PROCESSED / "week4_chunking_comparison.csv", index=False, encoding="utf-8-sig")
print("저장: week4_chunking_comparison.csv")
df_compare

---
## 10. 에러 케이스 분석 (전략별 최저 context_precision 질문)

In [ ]:
for name in STRATEGIES:
    df = score_tables[name]
    if "context_precision" not in df.columns:
        continue
    print("=" * 64)
    print(f"[{name}] context_precision 최저 2문항")
    worst = df.nsmallest(2, "context_precision")
    for _, r in worst.iterrows():
        fp = r.get("faithfulness", float("nan"))
        print(f"  질문: {r['question']}")
        print(f"  context_precision={r['context_precision']:.3f}  faithfulness={fp:.3f}")
        ctxs = r["contexts"] if isinstance(r["contexts"], list) else []
        for i, c in enumerate(ctxs[:2], 1):
            print(f"    [{i}] {str(c)[:150].replace(chr(10),' ')} ...")
        print()

---
## 11. retrospective 추가용 — 해석 가이드

아래 표를 `docs/week4_retrospective.md`의 **"7. 다음 단계"** 아래에 이어 붙인다.

| 지표 | A → B (size 효과) | B → C (클린+언어별) | 해석 방향 |
|---|---|---|---|
| Context Precision | ? | ? | 올랐다면 관련 없는 chunk가 줄었다 (노이즈 제거/size 효과) |
| Faithfulness | ? | ? | 올랐다면 깨끗한 context로 hallucination 감소 |
| Answer Relevancy | ? | ? | 큰 변화 없으면 검색 개선이 생성 단계까지 전달되지 않은 것 → 5주차 retrieval 고도화 필요 |

### 면접 대비 서사 (회고에 넘길 내용)
- A vs B: chunk_size만 바꿨을 때 수치 변화 → "한국어 문서가 512에서 잘려 맥락이 끊긴 문제를 size로 얼마나 해결했는가"
- B vs C: 동일 size에서 노이즈 제거 + 언어별 size가 추가로 얼마나 기여했는가
- 주의: C가 항상 이기는 것은 아니다. **지면 결과 자체가 서사**이며, "왜 그런 결과가 나왔는지" 가설을 설명하는 것이 포트폴리오 핵심

### 메타데이터 (4주차 5번 과제)
이미 loader에서 `page / language / org / doc_type / priority`가 붙어 있다. 전략 C의 언어별 차등 chunking이 `language` 메타를 활용한 첫 사례 → 5주차 Self-Query Retriever 빌드업으로 연결.